<a href="https://colab.research.google.com/github/Rifqi399/Pemprograman-dasar/blob/main/Rifqi(P9)Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Tugas 2: A Modul KPI Pembangkit Listrik
# RPE311 BASIC PROGRAMMING - Week 7

def heat_rate(coal_t, calorific_value, energy_mwh):
    """Hitung heat rate dalam kcal/kWh; energy harus lebih besar dari nol."""
    if energy_mwh <= 0:
        raise ValueError("energy_mwh harus lebih besar dari 0")
    return coal_t * calorific_value / energy_mwh


def thermal_efficiency(heat_rate_kcal_kwh):
    """Hitung efisiensi termal dalam persen dari heat rate."""
    if heat_rate_kcal_kwh <= 0:
        raise ValueError("heat_rate_kcal_kwh harus lebih besar dari 0")
    return 860 / heat_rate_kcal_kwh * 100


def capacity_factor(energy_mwh, installed_mw, hours):
    """Hitung capacity factor dalam persen untuk periode yang diberikan."""
    if energy_mwh <= 0:
        raise ValueError("energy_mwh harus lebih besar dari 0")
    if installed_mw <= 0 or hours <= 0:
        raise ValueError("installed_mw dan hours harus lebih besar dari 0")
    return energy_mwh / (installed_mw * hours) * 100


def specific_coal_consumption(coal_t, energy_mwh):
    """Hitung konsumsi batubara spesifik dalam ton per MWh."""
    if energy_mwh <= 0:
        raise ValueError("energy_mwh harus lebih besar dari 0")
    return coal_t / energy_mwh


def day_status(efficiency, run_hours):
    """Kembalikan status: NOT FULL, LOW EFFICIENCY, atau NORMAL."""
    if run_hours < 24:
        return "NOT FULL"
    elif efficiency < 36.3:
        return "LOW EFFICIENCY"
    return "NORMAL"


# Assert tests dengan nilai contoh yang dihitung manual
assert abs(heat_rate(1, 4400, 2) - 2200) < 0.001
assert abs(thermal_efficiency(2200) - (860 / 2200 * 100)) < 0.001
assert abs(capacity_factor(165, 330, 24) - (165 / (330 * 24) * 100)) < 0.001
assert abs(specific_coal_consumption(1, 2) - 0.5) < 0.001
assert day_status(36.5, 24) == "NORMAL"
assert day_status(36.2, 24) == "LOW EFFICIENCY"
assert day_status(36.5, 18) == "NOT FULL"

# Uji penanganan energi nol
for call in [lambda: heat_rate(1, 4400, 0),
             lambda: capacity_factor(0, 330, 24),
             lambda: specific_coal_consumption(1, 0)]:
    try:
        call()
        raise AssertionError("Seharusnya ValueError untuk energi nol")
    except ValueError:
        pass


def print_report(energy, coal, run_hours, installed_mw, calorific_value, title):
    """Cetak tabel KPI harian untuk satu unit pembangkit."""
    print("\n" + title)
    print(f"{'Day':>3} {'HR kcal/kWh':>12} {'Eff %':>9} {'CF %':>9} {'t/MWh':>9} {'Status':<15}")
    efficiencies = []
    statuses = []
    for day, (e, c, hours) in enumerate(zip(energy, coal, run_hours), start=1):
        hr = heat_rate(c, calorific_value, e)
        eff = thermal_efficiency(hr)
        cf = capacity_factor(e, installed_mw, hours)
        scc = specific_coal_consumption(c, e)
        status = day_status(eff, hours)
        efficiencies.append(eff)
        statuses.append(status)
        print(f"{day:>3} {hr:>12,.2f} {eff:>9.2f} {cf:>9.2f} {scc:>9.3f} {status:<15}")
    return efficiencies, statuses


# Assert untuk prosedur laporan
test_eff, test_status = print_report([2], [1], [24], 330, 4400, "TEST REPORT")
assert len(test_eff) == 1 and len(test_status) == 1
assert test_status[0] == "NORMAL"


# Data Unit 1: 1-30 November 2026
energy = [5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
          6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
          5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062] # MWh
coal = [2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
        3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
        3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240] # tonnes
run_hours = [24] * 30
run_hours[16] = 18  # Day 17: unit ran only 18 hours

unit1_eff, unit1_status = print_report(
    energy, coal, run_hours, installed_mw=330,
    calorific_value=4400, title="UNIT 1 - NOVEMBER 2026"
)

# Ringkasan Unit 1
average_efficiency = sum(unit1_eff) / len(unit1_eff)
monthly_capacity_factor = sum(energy) / (330 * sum(run_hours)) * 100
status_counts = {s: unit1_status.count(s) for s in ["NOT FULL", "LOW EFFICIENCY", "NORMAL"]}
least_efficient_days = sorted(range(1, 31), key=lambda d: unit1_eff[d - 1])[:3]

print("\nUNIT 1 SUMMARY")
print(f"Average efficiency        : {average_efficiency:.2f} %")
print(f"Monthly capacity factor   : {monthly_capacity_factor:.2f} %")
print(f"Days per status           : {status_counts}")
print(f"Three least efficient days: {least_efficient_days}")


# Unit 2: gunakan fungsi yang sama, tanpa menyalin formula
energy_unit2 = [12150, 12080, 11020, 12210, 12095] # MWh
coal_unit2 = [5240, 5225, 4790, 5260, 5230] # tonnes
run_hours_unit2 = [24] * 5

unit2_eff, unit2_status = print_report(
    energy_unit2, coal_unit2, run_hours_unit2, installed_mw=600,
    calorific_value=5100, title="UNIT 2 - FIVE DAYS"
)

unit2_status_counts = {s: unit2_status.count(s) for s in ["NOT FULL", "LOW EFFICIENCY", "NORMAL"]}
print("\nUNIT 2 SUMMARY")
print(f"Average efficiency : {sum(unit2_eff) / len(unit2_eff):.2f} %")
print(f"Capacity factor    : {sum(energy_unit2) / (600 * sum(run_hours_unit2)) * 100:.2f} %")
print(f"Days per status    : {unit2_status_counts}")



TEST REPORT
Day  HR kcal/kWh     Eff %      CF %     t/MWh Status         
  1     2,200.00     39.09      0.03     0.500 NORMAL         

UNIT 1 - NOVEMBER 2026
Day  HR kcal/kWh     Eff %      CF %     t/MWh Status         
  1     2,365.71     36.35     70.40     0.538 NORMAL         
  2     2,350.80     36.58     76.99     0.534 NORMAL         
  3     2,352.75     36.55     76.74     0.535 NORMAL         
  4     2,355.57     36.51     76.96     0.535 NORMAL         
  5     2,353.57     36.54     76.69     0.535 NORMAL         
  6     2,350.24     36.59     76.73     0.534 NORMAL         
  7     2,366.86     36.34     70.25     0.538 NORMAL         
  8     2,364.05     36.38     70.10     0.537 NORMAL         
  9     2,350.03     36.60     77.02     0.534 NORMAL         
 10     2,356.60     36.49     76.81     0.536 NORMAL         
 11     2,347.58     36.63     76.98     0.534 NORMAL         
 12     2,347.77     36.63     76.88     0.534 NORMAL         
 13     2,353.05  